# Lezione 59 — Monitoraggio e drift

> **Modulo:** capstone · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 58 (pipeline), Lezione 51 (rischi online).
>
> Obiettivo pratico unico: monitorare il **drift** — quando i dati in ingresso si
> allontanano da quelli su cui il lab e' stato costruito — e far scattare un
> allarme, in NumPy.

## Teoria essenziale

Un modello in produzione non e' "finito": il mondo cambia. Il **drift** e' lo
spostamento della distribuzione dei dati in ingresso rispetto al **riferimento**
(il train). Se non lo si misura, la qualita' cala in silenzio. Monitoriamo due
segnali semplici e leggibili:

- **drift di distribuzione delle classi**: la proporzione dei tipi cambia?
- **drift di una feature**: la lunghezza media dei testi cambia?

Una metrica robusta e' il **Population Stability Index (PSI)**: somma su ogni
bin di $(p_{now}-p_{ref})\cdot\log(p_{now}/p_{ref})$. Convenzione comune: PSI
< 0.1 stabile, 0.1-0.25 moderato, > 0.25 forte.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

proc = Path('..') / 'datasets' / 'processed'
TIPI = ["episodic", "semantic", "preference"]
_idx = {t: i for i, t in enumerate(TIPI)}
_train = pd.read_csv(proc / 'memory_train.csv')
_train = _train[_train['type'].isin(TIPI)].reset_index(drop=True)

def _tok(t):
    return re.findall(r"[a-zA-Z]+", str(t).lower())

# --- classificatore bag-of-words + softmax (Lezione 54) ---
_vocab = {}
for _t in _train['text']:
    for _w in _tok(_t):
        _vocab.setdefault(_w, len(_vocab))

def _bow(testo):
    v = np.zeros(len(_vocab))
    for w in _tok(testo):
        if w in _vocab:
            v[_vocab[w]] += 1.0
    return v

def _softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

_Xtr = np.vstack([_bow(t) for t in _train['text']])
_ytr = np.array([_idx[t] for t in _train['type']])
_W = np.zeros((len(_vocab), len(TIPI)))
_b = np.zeros(len(TIPI))
_Y = np.eye(len(TIPI))[_ytr]
for _ in range(300):
    _P = _softmax(_Xtr @ _W + _b)
    _W -= 0.5 * (_Xtr.T @ (_P - _Y) / len(_Xtr) + 1e-3 * _W)
    _b -= 0.5 * (_P - _Y).mean(axis=0)

def classifica_tipo(testo):
    return TIPI[int(_softmax(_bow(testo)[None, :] @ _W + _b).argmax())]

# --- rappresentazione: embedding, entita' (Lezione 55) ---
DIM = 48
def embed(testo):
    v = np.zeros(DIM)
    for w in _tok(testo):
        v[int.from_bytes(w.encode(), 'little') % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def estrai_entita(testo):
    return [w for w in re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo)) if w not in {"The", "A", "User"}]

# --- relazioni: fallback a regole (Lezione 56) ---
_VERBI = {"visited": "visited", "met": "met", "likes": "likes",
          "prefers": "prefers", "works": "works_at", "lives": "lives_in"}
def estrai_relazioni(testo):
    ent = estrai_entita(testo)
    for v_sup, v_norm in _VERBI.items():
        if v_sup in testo.lower() and len(ent) >= 2:
            return [{"source": ent[0], "type": v_norm, "target": ent[1]}]
    return []

# --- importanza composita (Lezione 25, versione compatta) ---
_FORTI = {"prefers", "likes", "dislikes", "hates", "loves", "always", "never", "important"}
def calcola_importanza(testo):
    # composita (Lezione 25): lunghezza + parole forti + numero di entita'
    parole = _tok(testo)
    forti = sum(1 for w in parole if w in _FORTI)
    lung = min(len(parole) / 15.0, 1.0)
    n_ent = len(estrai_entita(testo))
    imp = 0.30 * lung + 0.40 * min(forti, 1) + 0.20 * min(n_ent, 2) / 2 + 0.10
    return round(float(min(imp, 1.0)), 2)

In [2]:
rng = np.random.default_rng(59)

def distrib_tipi(testi):
    conteggi = np.zeros(len(TIPI))
    for t in testi:
        conteggi[_idx[classifica_tipo(t)]] += 1
    return conteggi / max(conteggi.sum(), 1)

def psi(p_ref, p_now, eps=1e-4):
    p_ref = np.clip(p_ref, eps, None)
    p_now = np.clip(p_now, eps, None)
    return float(np.sum((p_now - p_ref) * np.log(p_now / p_ref)))

# riferimento = distribuzione dei tipi nel train
p_ref = distrib_tipi(_train['text'])
print("distribuzione di riferimento (episodic/semantic/preference):", np.round(p_ref, 2))

distribuzione di riferimento (episodic/semantic/preference): [0.51 0.2  0.29]


In [3]:
# batch 'in produzione' 1: simile al riferimento (campiono dal train)
batch_ok = _train['text'].sample(40, random_state=1).tolist()
# batch 2: drift — quasi tutte preferenze (il mondo e' cambiato)
solo_pref = _train[_train['type'] == 'preference']['text']
batch_drift = solo_pref.sample(40, replace=True, random_state=2).tolist()

for nome, batch in [("batch simile", batch_ok), ("batch con drift", batch_drift)]:
    p_now = distrib_tipi(batch)
    valore = psi(p_ref, p_now)
    stato = "OK" if valore < 0.1 else ("MODERATO" if valore < 0.25 else "ALLARME")
    print(f"{nome:16}: distrib {np.round(p_now,2)}  PSI={valore:.3f} -> {stato}")

batch simile    : distrib [0.42 0.2  0.38]  PSI=0.038 -> OK
batch con drift : distrib [0. 0. 1.]  PSI=6.779 -> ALLARME


Leggi i due PSI: il batch simile resta sotto soglia; il batch con drift (quasi
solo preferenze) supera la soglia di allarme. E' il segnale per intervenire —
raccogliere nuovo feedback (Fase 7) e ri-tarare i componenti.

In [4]:
def monitor(batch, p_ref=p_ref, soglia=0.25):
    valore = psi(p_ref, distrib_tipi(batch))
    return {"psi": round(valore, 3), "allarme": valore >= soglia}

m_ok = monitor(batch_ok)
m_drift = monitor(batch_drift)
# controlli di non-regressione
assert not m_ok["allarme"]
assert m_drift["allarme"]
print("monitor batch simile:", m_ok)
print("monitor batch drift :", m_drift)

monitor batch simile: {'psi': 0.038, 'allarme': False}
monitor batch drift : {'psi': 6.779, 'allarme': True}


## Riepilogo (max 8 punti)

1. Un modello in produzione non e' "finito": il mondo cambia.
2. **Drift** = la distribuzione in ingresso si allontana dal riferimento.
3. Senza misurarlo, la qualita' cala **in silenzio**.
4. Segnali semplici: distribuzione delle classi, feature medie.
5. **PSI** misura lo scostamento tra due distribuzioni.
6. Convenzione: <0.1 stabile, 0.1-0.25 moderato, >0.25 forte.
7. Oltre soglia scatta l'**allarme**: raccogliere feedback e ri-tarare.
8. Il monitoraggio chiude il ciclo con la Fase 7.

## Quiz

1. Perche' un modello puo' peggiorare senza che il codice cambi?
2. Cosa misura il PSI?
3. Cosa si fa quando scatta l'allarme di drift?

*(Risposte: 1. perche' la distribuzione dei dati in ingresso cambia (drift);
2. lo scostamento tra la distribuzione attuale e quella di riferimento; 3. si
raccoglie nuovo feedback e si ri-tarano/ri-addestrano i componenti.)*